<a href="https://colab.research.google.com/github/kkk-boop/network-analyser/blob/main/network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Anormaly detection using k mean to detect suspicious fire wall**

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import itertools

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, confusion_matrix
)
from sklearn.model_selection import GridSearchCV
from scipy.stats import zscore
from joblib import Parallel, delayed

# Load the Firewall dataset
df = pd.read_csv("Firewall.csv")

# Optional: Preview the data
print("Firewall dataset loaded successfully:")
print(df.head())

# Define new rows to add (must match existing column names)
new_rows = [
    {
        'Source Port': 12345,
        'Destination Port': 80,
        'NAT Source Port': 12345,
        'NAT Destination Port': 80,
        'Bytes': 7000,
        'Bytes Sent': 5000,
        'Bytes Received': 2000,
        'Packets': 25,
        'Elapsed Time (sec)': 5.5,
        'pkts_sent': 13,
        'pkts_received': 12,
        'Action': 'deny'
    },
    {
        'Source Port': 6789,
        'Destination Port': 443,
        'NAT Source Port': 6789,
        'NAT Destination Port': 443,
        'Bytes': 300,
        'Bytes Sent': 150,
        'Bytes Received': 150,
        'Packets': 3,
        'Elapsed Time (sec)': 1.0,
        'pkts_sent': 1,
        'pkts_received': 2,
        'Action': 'allow'
    }
]

# Append the new rows to the existing DataFrame
df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

# Check updated data
print("Updated Firewall dataset with new rows:")
print(df.tail())



FileNotFoundError: [Errno 2] No such file or directory: 'Firewall.csv'

In [ ]:

# One-hot encode the 'Action' column
encoder = OneHotEncoder(sparse_output=False)
encoded_action = encoder.fit_transform(df[['Action']])
encoded_action_df = pd.DataFrame(encoded_action, columns=encoder.get_feature_names_out(['Action']))

# Display the first few rows of the one-hot encoded dataframe
print("\nOne-hot encoded 'Action' columns:")
print(encoded_action_df)

In [ ]:
 # Concatenate one-hot encoded 'Action' columns into the original DataFrame
sampled_df = pd.concat([df, encoded_action_df], axis=1)

# Define feature columns: numerical + one-hot encoded 'Action' columns
base_features = [
    'Source Port', 'Destination Port', 'NAT Source Port', 'NAT Destination Port',
    'Bytes', 'Bytes Sent', 'Bytes Received', 'Packets',
    'Elapsed Time (sec)', 'pkts_sent', 'pkts_received'
]
encoded_action_features = list(encoded_action_df.columns)

# Final feature set
features = base_features + encoded_action_features

# Optional: Display first few rows for verification
print("Sampled DataFrame with encoded 'Action' features:")
print(sampled_df.head())


In [ ]:
# Z-score method to identify outliers
z_threshold = 2  # Lowered threshold to make the method more sensitive
z_scores = np.abs(zscore(sampled_df[features]))
z_outliers = np.where(z_scores > z_threshold)

# IQR method to identify outliers
Q1 = sampled_df[features].quantile(0.25)
Q3 = sampled_df[features].quantile(0.75)
IQR = Q3 - Q1
iqr_multiplier = 3  # Increased multiplier to make the method more sensitive
iqr_outliers = ((sampled_df[features] < (Q1 - iqr_multiplier * IQR)) | (sampled_df[features] > (Q3 + iqr_multiplier * IQR))).any(axis=1)

In [ ]:
# Display outliers identified by Z-score
print("Outliers identified by Z-score method with threshold", z_threshold)
z_outlier_indices = np.unique(z_outliers[0])
print(sampled_df.iloc[z_outlier_indices])


In [ ]:
# Display outliers identified by IQR method
print("Outliers identified by IQR method with multiplier", iqr_multiplier)
print(df[iqr_outliers])


In [ ]:
# Function to remove outliers
def remove_outliers(data, method='zscore', threshold=2, multiplier=3):
    if method == 'zscore':
        z_scores = np.abs(zscore(data[features]))
        return data[(z_scores < threshold).all(axis=1)]
    elif method == 'iqr':
        Q1 = data[features].quantile(0.25)
        Q3 = data[features].quantile(0.75)
        IQR = Q3 - Q1
        return data[~((data[features] < (Q1 - multiplier * IQR)) | (data[features] > (Q3 + multiplier * IQR))).any(axis=1)]

# Example of removing outliers using adjusted Z-score method
cleaned_data_zscore = remove_outliers(sampled_df, method='zscore', threshold=z_threshold)

# Example of removing outliers using adjusted IQR method
cleaned_data_iqr = remove_outliers(sampled_df, method='iqr', multiplier=iqr_multiplier)

# Save the cleaned data if needed
cleaned_data_zscore.to_csv('cleaned_data_zscore_adjusted.csv', index=False)
cleaned_data_iqr.to_csv('cleaned_data_iqr_adjusted.csv', index=False)


# Display results
print("Cleaned Data (Z-Score Method, Adjusted):")
print(cleaned_data_zscore.head())
print(cleaned_data_zscore.shape)

print("Cleaned Data (IQR Method, Adjusted):")
print(cleaned_data_iqr.head())
print(cleaned_data_iqr.shape)

In [ ]:
#Due to complexity we will only use X_sampled
cleaned_data_zscore = pd.read_csv('cleaned_data_zscore_adjusted.csv')
# Assuming 'sampled_df' and 'features' are already defined
X_sampled = cleaned_data_zscore[features].astype(float)  # Ensure features are in the correct type

# Scale the features
scaler = StandardScaler()
X_sampled_scaled = scaler.fit_transform(X_sampled)

# Reduce the data size for faster computation
# Adjust the size of the sample as needed
X_sampled_reduced = X_sampled.sample(frac=0.5, random_state=42)

# Scale the features
scaler = StandardScaler()
X_sampled_scaled = scaler.fit_transform(X_sampled_reduced)

# Define hyperparameter grid
param_grid = {
    'n_clusters': [2, 3, 4],  # Slightly expanded for better tuning
    'init': ['k-means++'],
    'max_iter': [300],
    'tol': [1e-4, 1e-3],  # Include two tolerances
    'n_init': [10, 15]  # Include two n_init values
}

# Generate all combinations of hyperparameters
param_combinations = list(itertools.product(
    param_grid['n_clusters'],
    param_grid['init'],
    param_grid['max_iter'],
    param_grid['tol'],
    param_grid['n_init']
))

# Define a function to evaluate a set of hyperparameters
def evaluate_params(params):
    n_clusters, init, max_iter, tol, n_init = params
    kmeans = KMeans(
        n_clusters=n_clusters,
        init=init,
        max_iter=max_iter,
        tol=tol,
        n_init=n_init,
        random_state=42
    )
    labels = kmeans.fit_predict(X_sampled_scaled)
    if len(set(labels)) < 2:
        return -1  # Return a low score if only one cluster is found
    return silhouette_score(X_sampled_scaled, labels), params

# Measure the start time
start_time = time.time()

# Perform the manual grid search with parallel processing
results = Parallel(n_jobs=-1)(delayed(evaluate_params)(params) for params in param_combinations)

# Find the best score and corresponding parameters
best_score, best_params = max(results, key=lambda x: x[0])

# Measure the end time
end_time = time.time()
elapsed_time = end_time - start_time

# Output the best parameters and the time taken
print(f"Best Parameters: {best_params}")
print(f"Best Score: {best_score}")
print(f"Time Taken: {elapsed_time} seconds")

In [ ]:
# Unpack the best parameters
n_clusters, init, max_iter, tol, n_init = best_params

# Apply K-means clustering with the optimal hyperparameters on the full dataset
kmeans_optimal = KMeans(
    n_clusters=n_clusters,
    init=init,
    max_iter=max_iter,
    tol=tol,
    n_init=n_init,
    random_state=42
)

# Scale the full dataset
X_full_scaled = scaler.fit_transform(X_sampled)

# Fit and predict the clusters on the full dataset
cleaned_data_zscore["Cluster"] = kmeans_optimal.fit_predict(X_full_scaled)

# Evaluate the clustering using various metrics
silhouette_avg = silhouette_score(X_full_scaled, cleaned_data_zscore["Cluster"])
print(f"Silhouette Score: {silhouette_avg}")

In [ ]:

# Scale the full dataset
X_full_scaled = scaler.fit_transform(X_sampled)

# Fit and predict the clusters on the full dataset
cleaned_data_zscore["Cluster"] = kmeans_optimal.fit_predict(X_full_scaled)

# Now that we have the "Cluster" column in the dataframe, let's create the scatter plots

# List of significant features to plot against the common feature
significant_features = ['Bytes Sent', 'Bytes Received', 'Packets', 'Elapsed Time (sec)', 'pkts_sent', 'pkts_received']
common_feature = 'Bytes'  # Change this to any feature you want to compare against

# Create a custom palette with red for anomalies
palette = {1: 'blue', 0: 'red'}  # Assuming 0 means normal and 1 means abnormal

# Creating visualizations to highlight the anomalies
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(20, 25))

# Adjust the number of subplots based on the number of features
valid_axes = []

for ax, feature in zip(axes.flatten(), significant_features):
    if feature in cleaned_data_zscore.columns and not cleaned_data_zscore[feature].isnull().all():
        sns.scatterplot(x=common_feature, y=feature, hue='Cluster', data=cleaned_data_zscore, palette=palette, ax=ax)
        ax.set_title(f'Scatter plot of {common_feature} vs {feature}')
        valid_axes.append(ax)
    else:
        fig.delaxes(ax)

plt.tight_layout()
plt.show()

In [ ]:
# Identifying potential anomalies
normal_data = cleaned_data_zscore[cleaned_data_zscore['Cluster'] != 0]
anomalies = cleaned_data_zscore[cleaned_data_zscore['Cluster'] == 0]

In [ ]:
# Display the number of anomalies
num_anomalies = anomalies.shape[0]
print(f"Number of anomalies: {num_anomalies}")

In [ ]:
# Display anomalies in a readable table
from tabulate import tabulate
print(tabulate(anomalies, headers='keys', tablefmt='psql'))

**Using K Nearest Neighbour**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Assuming the last column is the target and others are features
# Extract features and target
X = cleaned_data_zscore[features].values
y = cleaned_data_zscore['Cluster'].values  # Assuming 'Cluster' is the target column

# Step 3: Preprocess data
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the features (optional but recommended)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 4: Initialize k-NN model
k = 5  # Number of neighbors
knn = KNeighborsClassifier(n_neighbors=k)

# Step 5: Train model
knn.fit(X_train, y_train)

# Step 6: Evaluate model
y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy * 100:.2f}%")


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

# Load example dataset
data = load_iris()
X = data.data
y = data.target

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train a model (example: RandomForestClassifier)
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Perform PCA for dimensionality reduction
pca = PCA(n_components=2)
X_test_pca = pca.fit_transform(X_test)

# Plotting
plt.figure(figsize=(12, 6))

# True labels scatter plot
plt.subplot(1, 2, 1)
sns.scatterplot(x=X_test_pca[:, 0], y=X_test_pca[:, 1], hue=y_test, palette='viridis', marker='o', alpha=0.6)
plt.title('True Labels')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

# Predicted labels scatter plot
plt.subplot(1, 2, 2)
sns.scatterplot(x=X_test_pca[:, 0], y=X_test_pca[:, 1], hue=y_pred, palette='viridis', marker='o', alpha=0.6)
plt.title('Predicted Labels')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrix visualization for y_test
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y_test), yticklabels=np.unique(y_test))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Count the number of anomalies detected
num_anomalies = np.sum(y_pred == 1)
print(f"Number of anomalies detected: {num_anomalies}")

In [ ]:
# Identify the anomalies
anomalies_indices = np.where(y_pred == 1)
anomalies = cleaned_data_zscore.iloc[anomalies_indices]

# Display the anomalies
anomalies

**Spectral clustering**


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import SpectralClustering
import pandas as pd
import numpy as np



# Encode the 'Action' column
label_encoder = LabelEncoder()
cleaned_data_zscore['Action'] = label_encoder.fit_transform(cleaned_data_zscore['Action'])

# Select relevant features for clustering (all columns except 'Action')
features = ['Source Port', 'Destination Port', 'NAT Source Port', 'NAT Destination Port',
            'Bytes', 'Bytes Sent', 'Bytes Received', 'Packets', 'Elapsed Time (sec)',
            'pkts_sent', 'pkts_received']

# Normalize the data
scaler = StandardScaler()
normalized_features = scaler.fit_transform(cleaned_data_zscore[features])

# Step 1: Perform spectral clustering on a smaller subset
sample_size = 2000  # Define a smaller sample size
smaller_sampled_data = cleaned_data_zscore.sample(n=sample_size, random_state=42)
smaller_sampled_normalized_features = scaler.transform(smaller_sampled_data[features])

spectral_clustering = SpectralClustering(n_clusters=5, affinity='nearest_neighbors', n_neighbors=10, random_state=42)
smaller_sampled_data['Cluster'] = spectral_clustering.fit_predict(smaller_sampled_normalized_features)

# Step 2: Extract features and target from the smaller subset
X = smaller_sampled_data[features].values
y = smaller_sampled_data['Cluster'].values

# Step 3: Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Step 4: Standardize the features
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 5: Initialize k-NN model
k = 5  # Number of neighbors
knn = KNeighborsClassifier(n_neighbors=k)

# Step 6: Train the k-NN classifier
knn.fit(X_train, y_train)

# Step 7: Evaluate the classifier
y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

accuracy

In [ ]:
from sklearn.cluster import SpectralClustering
import numpy as np

# Apply spectral clustering to the smaller sampled data
spectral_clustering_smaller_sampled = SpectralClustering(n_clusters=5, affinity='nearest_neighbors', n_neighbors=10, random_state=42)
smaller_sampled_clusters = spectral_clustering_smaller_sampled.fit_predict(smaller_sampled_normalized_features)

# Add cluster labels to the smaller sampled data
smaller_sampled_data['Cluster'] = smaller_sampled_clusters

# Identify outliers in the smaller sampled data
smaller_sampled_cluster_counts = np.bincount(smaller_sampled_clusters)
smaller_sampled_outlier_clusters = np.where(smaller_sampled_cluster_counts < 20)[0]  # clusters with fewer than 20 points

# Mark smaller sampled data points in these clusters as anomalies
smaller_sampled_data['Anomaly'] = smaller_sampled_data['Cluster'].apply(lambda x: 1 if x in smaller_sampled_outlier_clusters else 0)

# Display the results
cleaned_data_zscore.head()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.cluster import SpectralClustering
from sklearn.metrics import silhouette_score, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Hypothetical data generation for testing purpose
data = {
    'Source Port': np.random.randint(1000, 65000, size=2000),
    'Destination Port': np.random.randint(1000, 65000, size=2000),
    'NAT Source Port': np.random.randint(1000, 65000, size=2000),
    'NAT Destination Port': np.random.randint(1000, 65000, size=2000),
    'Bytes': np.random.randint(100, 10000, size=2000),
    'Bytes Sent': np.random.randint(100, 5000, size=2000),
    'Bytes Received': np.random.randint(100, 5000, size=2000),
    'Packets': np.random.randint(1, 100, size=2000),
    'Elapsed Time (sec)': np.random.randint(1, 1000, size=2000),
    'pkts_sent': np.random.randint(1, 100, size=2000),
    'pkts_received': np.random.randint(1, 100, size=2000),
    'Action': np.random.choice(['allow', 'deny'], size=2000)
}

firewall_data = pd.DataFrame(data)

# Encode the 'Action' column
label_encoder = LabelEncoder()
firewall_data['Action'] = label_encoder.fit_transform(firewall_data['Action'])

# Select relevant features for clustering (all columns except 'Action')
features = ['Source Port', 'Destination Port', 'NAT Source Port', 'NAT Destination Port',
            'Bytes', 'Bytes Sent', 'Bytes Received', 'Packets', 'Elapsed Time (sec)',
            'pkts_sent', 'pkts_received']

# Normalize the data
scaler = StandardScaler()
normalized_features = scaler.fit_transform(firewall_data[features])

# Perform spectral clustering
n_clusters = 5
n_neighbors = 10

spectral_clustering = SpectralClustering(
    n_clusters=n_clusters,
    affinity='nearest_neighbors',
    n_neighbors=n_neighbors,
    random_state=42
)
firewall_data['Cluster'] = spectral_clustering.fit_predict(normalized_features)

# Evaluate the clustering using the silhouette score
silhouette_avg = silhouette_score(normalized_features, firewall_data['Cluster'])
print(f"Silhouette Score: {silhouette_avg}")

# Compute confusion matrix
y_true = firewall_data['Cluster']
y_pred = firewall_data['Cluster']
cm = confusion_matrix(y_true, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y_true), yticklabels=np.unique(y_true))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Visualize the results using PCA for dimensionality reduction
pca = PCA(n_components=2)
features_pca = pca.fit_transform(normalized_features)

plt.figure(figsize=(12, 6))

# True labels scatter plot
plt.subplot(1, 2, 1)
sns.scatterplot(x=features_pca[:, 0], y=features_pca[:, 1], hue=y_true, palette='viridis', marker='o', alpha=0.6)
plt.title('True Labels')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

# Predicted labels scatter plot
plt.subplot(1, 2, 2)
sns.scatterplot(x=features_pca[:, 0], y=features_pca[:, 1], hue=y_pred, palette='viridis', marker='o', alpha=0.6)
plt.title('Predicted Labels')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

plt.tight_layout()
plt.show()


In [ ]:
# Count the number of anomalies detected
num_anomalies = np.sum(y_pred == 1)
print(f"Number of anomalies detected: {num_anomalies}")

In [ ]:
# Identify the anomalies
anomalies_indices = np.where(y_pred == 1)
anomalies = cleaned_data_zscore.iloc[anomalies_indices]

# Display the anomalies
anomalies